# 详细文档: Preissmann模型 - 河段

**相关模块:** `preissmann_model/reach.py`

## 目的
此文档旨在详细介绍`preissmann_model`中河段（`RiverReach`）的定义和使用。`RiverReach`对象是`HydraulicModel`的核心几何组成部分，它代表了由一系列计算节点（断面）连接而成的一段连续河道。

此笔记本将展示如何：
1.  使用前一节中定义的断面类（`RectangularCrossSection`, `TrapezoidalCrossSection`）来创建断面列表。
2.  结合河段长度、坡度、糙率等信息，创建一个完整的`RiverReach`实例。
3.  可视化创建的河段的纵剖面图。

## 1. 环境设置

首先，我们导入所需的库和我们自己的模型模块。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection, TrapezoidalCrossSection

## 2. 创建一个简单的均匀河段

我们首先创建一个最简单的河段：一个所有断面形状都相同的棱柱体河道。我们使用11个相同的矩形断面来定义这个河段。

In [ ]:
num_nodes = 11
cross_sections_uniform = [RectangularCrossSection(width=10) for _ in range(num_nodes)]
lengths_uniform = np.full(num_nodes - 1, 500.0) # 每个小段长度为500m

reach_uniform = RiverReach(
    cross_sections=cross_sections_uniform,
    lengths=lengths_uniform,
    slope=0.001,
    manning_n=0.03
)

print(f"创建了一个均匀河段，包含 {reach_uniform.num_sections} 个断面。")

### 可视化河段纵剖面

我们可以计算并绘制出河段的河床高程（`Z_bed`）。河床高程是根据河段的坡度 `slope` 和长度 `lengths` 从下游向上游推算的。通常，为了方便计算，模型假设最下游的河床高程为0。

In [ ]:
def get_bed_elevation(reach):
    z_bed = np.zeros(reach.num_sections)
    # 从下游向上游计算
    for i in range(reach.num_sections - 2, -1, -1):
        z_bed[i] = z_bed[i+1] + reach.lengths[i] * reach.slope
    return z_bed

def plot_reach_profile(ax, reach, title):
    z_bed = get_bed_elevation(reach)
    distances = np.zeros(reach.num_sections)
    distances[1:] = np.cumsum(reach.lengths)
    
    ax.plot(distances, z_bed, 'k-o', label='Bed Elevation')
    ax.set_title(title)
    ax.set_xlabel('Distance from Upstream (m)')
    ax.set_ylabel('Bed Elevation (m)')
    ax.grid(True)

fig, ax = plt.subplots(figsize=(10, 5))
plot_reach_profile(ax, reach_uniform, 'Longitudinal Profile of Uniform Reach')

## 3. 创建一个非均匀（复合）河段

在自然界中，河道断面很少是均匀的。`RiverReach`类也支持定义一个断面形状沿程变化的非棱柱体河道。例如，我们可以创建一个上游是梯形、下游是矩形的复合河道。

In [ ]:
num_nodes_compound = 15

# 上游5个断面是梯形
cs_upstream = [TrapezoidalCrossSection(bottom_width=8, side_slope=1.5) for _ in range(5)]
# 下游10个断面是矩形
cs_downstream = [RectangularCrossSection(width=12) for _ in range(10)]

cross_sections_compound = cs_upstream + cs_downstream
lengths_compound = np.full(num_nodes_compound - 1, 300.0)

reach_compound = RiverReach(
    cross_sections=cross_sections_compound,
    lengths=lengths_compound,
    slope=0.0008,
    manning_n=0.035
)

print(f"创建了一个复合河段，包含 {reach_compound.num_sections} 个断面。")

### 可视化复合河段

我们可以用同样的方式绘制出复合河段的纵剖面。虽然在这个图上看不出断面形状的变化，但在进行水力计算时，模型会在每个节点使用其对应的断面定义。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_reach_profile(ax, reach_compound, 'Longitudinal Profile of Compound Reach')